# 04 Baseline Model Experiments
**三切法 + Confidence Threshold + 多模型比較**

- Train 64% / Validation 16% / Test 20%
- 按 grid_id 分組，避免空間洩漏

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, precision_score
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")

PROC_DIR  = "../data/processed"
MODEL_DIR = "../outputs/models"
os.makedirs(MODEL_DIR, exist_ok=True)

CATEGORIES = ["violent", "property", "drug", "public_order", "other"]
CAT_FEATURES = ["time_slot", "is_weekend", "slot_x_weekend"]
print("Setup done")

## 1. 載入資料

In [ ]:
df = pd.read_csv(f"{PROC_DIR}/all_cities.csv", low_memory=False)
df = df.dropna(subset=["latitude","longitude","crime_category"])
print(f"總資料：{len(df):,} 筆")
print(df["city"].value_counts())

## 2. 特徵工程

In [ ]:
def build_grid_stats(df):
    d = df.copy()
    d["lat_bin"]   = (d["latitude"]  / 0.01).round() * 0.01
    d["lon_bin"]   = (d["longitude"] / 0.01).round() * 0.01
    d["time_slot"] = pd.cut(d["hour"], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(int)
    for cat in CATEGORIES:
        d[f"is_{cat}"] = (d["crime_category"] == cat).astype(int)
    grid = d.groupby(["lat_bin","lon_bin","time_slot"]).agg(
        total_count=("crime_category","size"),
        **{f"hist_{cat}": (f"is_{cat}","mean") for cat in CATEGORIES}
    ).reset_index()
    grid["log_count"] = np.log1p(grid["total_count"])
    return grid

def add_spatial_lag(grid):
    step = 0.01
    lags = []
    for _, row in grid.iterrows():
        lat, lon = row["lat_bin"], row["lon_bin"]
        mask = (
            grid["lat_bin"].between(lat-step*1.5, lat+step*1.5) &
            grid["lon_bin"].between(lon-step*1.5, lon+step*1.5) &
            ~((grid["lat_bin"]==lat) & (grid["lon_bin"]==lon))
        )
        nb = grid[mask]
        lags.append(nb["hist_violent"].mean() if len(nb) > 0 else 0)
    grid["spatial_lag_violent"] = lags
    return grid

FEATURE_COLS = [
    "lat_bin","lon_bin","lat_norm","lon_norm",
    "hour_sin","hour_cos","month_sin","month_cos",
    "weekday_sin","weekday_cos",
    "is_weekend","time_slot","slot_x_weekend",
    "log_count","spatial_lag_violent",
    "hist_violent","hist_property","hist_drug","hist_public_order","hist_other",
]

def make_features(df, grid_stats):
    d = df.copy()
    d["lat_bin"]   = (d["latitude"]  / 0.01).round() * 0.01
    d["lon_bin"]   = (d["longitude"] / 0.01).round() * 0.01
    d["time_slot"] = pd.cut(d["hour"], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(int)
    d["hour_sin"]    = np.sin(2*np.pi*d["hour"]   /24)
    d["hour_cos"]    = np.cos(2*np.pi*d["hour"]   /24)
    d["month_sin"]   = np.sin(2*np.pi*d["month"]  /12)
    d["month_cos"]   = np.cos(2*np.pi*d["month"]  /12)
    d["weekday_sin"] = np.sin(2*np.pi*d["weekday"]/7)
    d["weekday_cos"] = np.cos(2*np.pi*d["weekday"]/7)
    d["is_weekend"]  = (d["weekday"] >= 5).astype(int)
    d["slot_x_weekend"] = d["time_slot"] * (d["is_weekend"]+1)
    d["lat_norm"] = (d["lat_bin"]-d["lat_bin"].mean())/(d["lat_bin"].std()+1e-8)
    d["lon_norm"] = (d["lon_bin"]-d["lon_bin"].mean())/(d["lon_bin"].std()+1e-8)
    d = d.merge(grid_stats, on=["lat_bin","lon_bin","time_slot"], how="left")
    d[FEATURE_COLS] = d[FEATURE_COLS].fillna(0)
    for c in CAT_FEATURES:
        d[c] = d[c].astype(str)
    return d[FEATURE_COLS], d["crime_category"]

print("特徵函數定義完成")

In [ ]:
# 建格子統計（spatial lag 需幾分鐘）
grid_stats_all = {}
for city in df["city"].unique():
    print(f"[{city}] 建構格子統計...")
    gs = build_grid_stats(df[df["city"]==city].copy())
    print(f"  計算 spatial lag...")
    gs = add_spatial_lag(gs)
    grid_stats_all[city] = gs
    gs.to_csv(f"{MODEL_DIR}/grid_stats_{city.lower()}.csv", index=False)
    print(f"  完成：{len(gs):,} 格")

## 3. 三切法

In [ ]:
def three_way_split(X, y, groups, val_size=0.16, test_size=0.20, rs=42):
    """Train 64% / Val 16% / Test 20%，按 grid_id 分組"""
    gss1 = GroupShuffleSplit(1, test_size=test_size, random_state=rs)
    tv_idx, test_idx = next(gss1.split(X, y, groups=groups))
    gss2 = GroupShuffleSplit(1, test_size=val_size/(1-test_size), random_state=rs)
    rel_tr, rel_val = next(gss2.split(X.iloc[tv_idx], y[tv_idx], groups=pd.Series(groups).iloc[tv_idx]))
    train_idx = tv_idx[rel_tr]
    val_idx   = tv_idx[rel_val]
    print(f"  Train:{len(train_idx):,}  Val:{len(val_idx):,}  Test:{len(test_idx):,}")
    return train_idx, val_idx, test_idx

# 選城市（改這裡切換）
CITY = "NYC"
city_df = df[df["city"]==CITY].copy()
city_df = city_df[city_df["crime_category"].isin(CATEGORIES)]
gs = grid_stats_all[CITY]
X, y = make_features(city_df, gs)
le = LabelEncoder(); le.fit(CATEGORIES)
y_enc = le.transform(y)
city_df = city_df.copy()
city_df["lat_bin"] = (city_df["latitude"]/0.01).round()*0.01
city_df["lon_bin"] = (city_df["longitude"]/0.01).round()*0.01
groups = (city_df["lat_bin"].astype(str)+"_"+city_df["lon_bin"].astype(str)).values
train_idx, val_idx, test_idx = three_way_split(X, y_enc, groups)
X_train,X_val,X_test = X.iloc[train_idx],X.iloc[val_idx],X.iloc[test_idx]
y_train,y_val,y_test = y_enc[train_idx],y_enc[val_idx],y_enc[test_idx]
sw_train = compute_sample_weight("balanced", y=y_train)
print("資料切分完成")

## 4. 多模型訓練

In [ ]:
# CatBoost
from catboost import CatBoostClassifier, Pool
cat_idx = [FEATURE_COLS.index(c) for c in CAT_FEATURES]
model_cat = CatBoostClassifier(
    iterations=600, depth=8, learning_rate=0.05,
    loss_function="MultiClass", eval_metric="TotalF1",
    auto_class_weights="Balanced", cat_features=cat_idx,
    random_seed=42, verbose=100,
)
model_cat.fit(
    Pool(X_train,y_train,cat_features=cat_idx),
    eval_set=Pool(X_val,y_val,cat_features=cat_idx),
    early_stopping_rounds=50,
)
print("CatBoost done")

In [ ]:
# LightGBM
import lightgbm as lgb
def to_int_cats(X):
    X2 = X.copy()
    for c in CAT_FEATURES: X2[c] = X2[c].astype(int)
    return X2
model_lgb = lgb.LGBMClassifier(
    n_estimators=600, max_depth=8, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1,
)
model_lgb.fit(
    to_int_cats(X_train), y_train,
    eval_set=[(to_int_cats(X_val), y_val)],
    callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)],
)
print("LightGBM done")

In [ ]:
# XGBoost
import xgboost as xgb
model_xgb = xgb.XGBClassifier(
    n_estimators=600, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="mlogloss",
    random_state=42, n_jobs=-1, early_stopping_rounds=50,
)
model_xgb.fit(
    to_int_cats(X_train), y_train, sample_weight=sw_train,
    eval_set=[(to_int_cats(X_val), y_val)], verbose=100,
)
print("XGBoost done")

## 5. Test Set 評估

In [ ]:
def evaluate(model, Xte, yte, name, le, use_int=False):
    Xt = to_int_cats(Xte) if use_int else Xte
    y_pred = model.predict(Xt)
    if hasattr(y_pred,"flatten"): y_pred = y_pred.flatten()
    proba  = model.predict_proba(Xt)
    pm  = precision_score(yte, y_pred, average="macro",    zero_division=0)
    pw  = precision_score(yte, y_pred, average="weighted", zero_division=0)
    f1m = f1_score(yte, y_pred, average="macro", zero_division=0)
    print(f"
===== {name} =====")
    print(f"Precision macro={pm:.4f}  weighted={pw:.4f}  F1 macro={f1m:.4f}")
    print(classification_report(yte, y_pred, target_names=le.classes_, zero_division=0))
    return y_pred, proba, pm, pw, f1m

pc,prc,pm_c,pw_c,f1_c = evaluate(model_cat, X_test, y_test, "CatBoost", le)
pl,prl,pm_l,pw_l,f1_l = evaluate(model_lgb, X_test, y_test, "LightGBM", le, use_int=True)
px,prx,pm_x,pw_x,f1_x = evaluate(model_xgb, X_test, y_test, "XGBoost",  le, use_int=True)

## 6. Confidence Threshold → Precision 0.8+

In [ ]:
import matplotlib.pyplot as plt

def selective_precision(proba, y_test, name, thresholds=None):
    if thresholds is None:
        thresholds = [0.3,0.4,0.5,0.6,0.7,0.8,0.85,0.9]
    rows = []
    for t in thresholds:
        conf = proba.max(axis=1) >= t
        if conf.sum() == 0:
            rows.append({"threshold":t,"coverage":0,"precision_macro":0,"precision_weighted":0})
            continue
        pm = precision_score(y_test[conf], proba.argmax(axis=1)[conf], average="macro",    zero_division=0)
        pw = precision_score(y_test[conf], proba.argmax(axis=1)[conf], average="weighted", zero_division=0)
        rows.append({"threshold":t,"coverage":round(conf.mean(),3),
                     "precision_macro":round(pm,4),"precision_weighted":round(pw,4)})
    res = pd.DataFrame(rows)
    print(f"
===== {name} Selective Precision =====")
    print(res.to_string(index=False))
    return res

rc = selective_precision(prc, y_test, "CatBoost")
rl = selective_precision(prl, y_test, "LightGBM")
rx = selective_precision(prx, y_test, "XGBoost")

fig, ax = plt.subplots(figsize=(9,5))
for res,name,col in [(rc,"CatBoost","#378ADD"),(rl,"LightGBM","#1D9E75"),(rx,"XGBoost","#D85A30")]:
    ax.plot(res["coverage"], res["precision_macro"], marker="o", label=name, color=col, lw=2)
ax.axhline(0.8, color="gray", ls="--", alpha=0.6, label="Precision=0.8")
ax.set_xlabel("Coverage（有多少樣本被預測）")
ax.set_ylabel("Precision macro")
ax.set_title(f"[{CITY}] Precision–Coverage Tradeoff")
ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(f"../outputs/eda/precision_coverage_{CITY.lower()}.png", dpi=150)
plt.show()

## 7. 儲存最佳模型

In [ ]:
import joblib
summary = pd.DataFrame([
    {"model":"CatBoost", "precision_macro":pm_c, "precision_weighted":pw_c, "f1_macro":f1_c},
    {"model":"LightGBM", "precision_macro":pm_l, "precision_weighted":pw_l, "f1_macro":f1_l},
    {"model":"XGBoost",  "precision_macro":pm_x, "precision_weighted":pw_x, "f1_macro":f1_x},
])
print(summary.to_string(index=False))
best = summary.loc[summary["precision_macro"].idxmax(),"model"]
print(f"
最佳模型：{best}")
if best=="CatBoost":  model_cat.save_model(f"{MODEL_DIR}/model_{CITY.lower()}_best.cbm")
elif best=="LightGBM": joblib.dump(model_lgb, f"{MODEL_DIR}/model_{CITY.lower()}_best_lgb.pkl")
else: model_xgb.save_model(f"{MODEL_DIR}/model_{CITY.lower()}_best.json")
print("儲存完成")